# QDM Map Alignment — Batch

Align **multiple** target Bz maps to a common reference coordinate frame.
Each sub-folder should contain `LED.jpg` and a Bz file (`.mat`, `.npy`, or `.csv`).

**Three-tier alignment per step:**

| Tier | Method | When |
|------|--------|------|
| 1 | ORB on raw LEDs | Always tried first |
| 2 | ORB with filter cycling (Sobel / Laplacian / Unsharp / CLAHE) | If tier 1 inliers < `MIN_INLIERS` |
| 3 | Manual tie-point clicking | If tier 2 also fails — pauses for your input |

Place `alignment_functions.py` next to this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2
import os
from pathlib import Path

from alignment_functions import (
    compute_affine_matrix,
    compute_affine_matrix_enhanced,
    compute_affine_matrix_manual,
    apply_affine,
    load_field_data,
    save_field_data,
)

---
## 1 — Configuration

In [ ]:
# ──────────────────────────────────────────────
# REFERENCE (coordinate frame you align TO)
# ──────────────────────────────────────────────
REFERENCE_LED = "./Testing_data/AF0/LED.jpg"


# ──────────────────────────────────────────────
# BATCH directory
# ──────────────────────────────────────────────
BATCH_DIR = "./Testing_data"

# ──────────────────────────────────────────────
# Output
# ──────────────────────────────────────────────
OUTPUT_DIR = "./aligned_output_batch_alignment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Enable manual fallback? Set False to skip failed steps instead of pausing.
ENABLE_MANUAL_FALLBACK = True

# ──────────────────────────────────────────────
# Data loading options
# ──────────────────────────────────────────────
MAT_KEY   = "Bz"       # variable name inside .mat files
SCALE     = 1e9        # conversion factor (T -> nT); set 1.0 if already in nT
FLIPUD    = True       # flip Bz vertically (QDM convention)

# ──────────────────────────────────────────────
# Plot options
# ──────────────────────────────────────────────
PIXEL_SIZE_UM = 2.35   # QDM pixel size in um (e.g. 2.35); None = axes in pixels
VMIN = -10000            # colour scale min (e.g. -10000); None = data min
VMAX = 10000            # colour scale max (e.g.  10000); None = data max

# ──────────────────────────────────────────────
# Alignment tuning
# ──────────────────────────────────────────────
N_FEATURES   = 500     # ORB features to detect
N_MATCHES    = 100     # max matches for affine estimation
MIN_INLIERS  = 30      # below this, move to next tier

# Tier 2: enhanced filter cycling
ENHANCE_SIGMA   = 1.0  # blur sigma for Sobel, Laplacian, Unsharp filters
ENHANCE_FILTERS = ["sobel", "laplacian", "unsharp", "clahe"]

# Tier 3: manual tie-points (if both auto tiers fail)
N_TIE_POINTS = 6       # number of points to click (minimum 3)

# ──────────────────────────────────────────────
# Output format
# ──────────────────────────────────────────────
SAVE_EXT  = ".mat"     # ".mat" | ".npy" | ".csv"

---
## 2 — Run batch alignment

**If `ENABLE_MANUAL_FALLBACK = True`**, the loop will pause on failed steps
and open interactive windows for you to click tie-points.

**Important**: if manual fallback is enabled, switch backend first:
```python
%matplotlib tk
```

In [ ]:
batch_root = Path(BATCH_DIR)
step_dirs = sorted([d for d in batch_root.iterdir() if d.is_dir()])

FIELD_EXTS = [".mat", ".npy", ".csv"]
ref_led = mpimg.imread(REFERENCE_LED)

print(f"Found {len(step_dirs)} step directories under {BATCH_DIR}\n")

results = {}

for step_dir in step_dirs:
    step_name = step_dir.name

    led_path = step_dir / "LED.jpg"
    if not led_path.exists():
        print(f"  [{step_name}] No LED.jpg — skipped.")
        continue

    field_path = None
    for ext in FIELD_EXTS:
        candidates = list(step_dir.glob(f"Bz*{ext}"))
        if candidates:
            field_path = candidates[0]
            break
    if field_path is None:
        print(f"  [{step_name}] No Bz file found — skipped.")
        continue

    print(f"{'='*50}")
    print(f"  {step_name}")
    print(f"{'='*50}")

    tgt_led = mpimg.imread(str(led_path))
    Bz = load_field_data(str(field_path), mat_key=MAT_KEY, scale=SCALE, flipud=FLIPUD)

    bz_size = (Bz.shape[1], Bz.shape[0])
    ref_led_r = cv2.resize(ref_led, bz_size, interpolation=cv2.INTER_AREA)
    tgt_led_r = cv2.resize(tgt_led, bz_size, interpolation=cv2.INTER_AREA)

    # ── Tier 1 ──
    M, aligned_led, n_inliers = compute_affine_matrix(
        ref_led_r, tgt_led_r,
        n_features=N_FEATURES, n_matches=N_MATCHES,
        sample_name=step_name, save_path=OUTPUT_DIR, show=True
    )
    method = "raw_ORB"

    # ── Tier 2 ──
    if M is None or n_inliers < MIN_INLIERS:
        print(f"  Tier 1: {n_inliers} inliers — trying enhanced filters")
        M, aligned_led, n_inliers = compute_affine_matrix_enhanced(
            ref_led_r, tgt_led_r,
            sigma=ENHANCE_SIGMA,
            n_features=N_FEATURES, n_matches=N_MATCHES,
            min_inliers=MIN_INLIERS,
            filters=ENHANCE_FILTERS,
            sample_name=f"{step_name}_enhanced", save_path=OUTPUT_DIR, show=True
        )
        method = "enhanced_ORB"

    # ── Tier 3: manual ──
    if (M is None or n_inliers < MIN_INLIERS) and ENABLE_MANUAL_FALLBACK:
        print(f"\n  Tier 2 failed for {step_name} — switching to manual tie-points")
        M, aligned_led, n_inliers = compute_affine_matrix_manual(
            ref_led_r, tgt_led_r,
            n_points=N_TIE_POINTS,
            sample_name=f"{step_name}_manual", save_path=OUTPUT_DIR, show=True
        )
        method = "manual"

    if M is None:
        print(f"  [{step_name}] All tiers failed — skipped.")
        continue

    print(f"  [{step_name}] Success: {method} ({n_inliers} inliers/points)")
    Bz_aligned = apply_affine(
        Bz, M, sample_name=step_name,
        vmin=VMIN, vmax=VMAX, pixel_size_um=PIXEL_SIZE_UM,
        save_path=OUTPUT_DIR, show=True
    )

    out_path = os.path.join(OUTPUT_DIR, f"{step_name}_Bz_aligned{SAVE_EXT}")
    save_field_data(out_path, Bz_aligned)

    results[step_name] = {"M": M, "shape": Bz_aligned.shape,
                          "inliers": n_inliers, "method": method}

print(f"\nDone — {len(results)}/{len(step_dirs)} steps aligned successfully.")

---
## 3 — Summary

In [ ]:
if results:
    print(f"{'Step':<15} {'Method':<16} {'Inliers':>8} {'Tx (px)':>10} {'Ty (px)':>10} {'Rot (deg)':>12}")
    print("-" * 73)
    for name, r in results.items():
        M = r["M"]
        tx, ty = M[0, 2], M[1, 2]
        angle = np.degrees(np.arctan2(M[1, 0], M[0, 0]))
        print(f"{name:<15} {r['method']:<16} {r['inliers']:>8d} {tx:>10.2f} {ty:>10.2f} {angle:>12.4f}")
else:
    print("No results. Check paths and run section 2.")